# GrammarFlow × Unlimited-OCR: small GPU trial

This is an interactive experiment using your Colab allowance, not a public API or a paid customer service. It does not change GrammarFlow.

1. In Colab, choose **Runtime → Change runtime type → GPU**, preferably the A100 offered by your plan.
2. Run the cells in order. Start with the synthetic sample. GPU setup/download/inference consume compute units; access is subject to your account balance and availability.
3. Download the results, then **Runtime → Disconnect and delete runtime** to stop idle GPU usage.

The model is pinned to a specific public revision and uses `trust_remote_code=True`, which executes that revision's Python code. It downloads about 6.7 GB of weights, plus dependencies/cache; no API key is needed. No Drive mount, tunnel, public server, or GrammarFlow credentials are used.

[Official setup](https://github.com/baidu/Unlimited-OCR#inference) · [Pinned model code](https://huggingface.co/baidu/Unlimited-OCR-2/tree/c14071b549f4fcab10909f3c4acc6a2edb366947) · [Colab limits](https://research.google.com/colaboratory/faq.html)


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
import torch

os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
ROOT = Path('/content/grammarflow_ocr')
ROOT.mkdir(exist_ok=True)
if not torch.cuda.is_available():
    raise RuntimeError('No GPU assigned. Choose Runtime → Change runtime type → GPU, then reconnect.')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('This model trial uses BF16. Choose a supported GPU such as A100; do not run CPU fallback.')
if shutil.disk_usage('/content').free < 18 * 1024**3:
    raise RuntimeError('At least 18 GiB of free disk is reserved for weights, cache and trial outputs.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('This checks assigned hardware, not your remaining compute-unit balance. Check that in Colab.')


## Install the small trial dependencies
Keep Colab's existing GPU-enabled PyTorch and matching torchvision. Upstream tested Python 3.10 / PyTorch 2.13.0; this notebook records the actual runtime instead of silently replacing its CUDA stack. Compatibility still needs a real Colab run. The model runs in a fresh child process, so installing Transformers does not require reusing imports cached in the notebook kernel.


In [ ]:
packages = [
    'transformers==4.52.1', 'huggingface-hub==0.36.0', 'tokenizers==0.21.4',
    'addict==2.4.0', 'einops==0.8.2', 'easydict==1.13', 'accelerate==1.7.0',
    'pymupdf==1.27.2.2', 'pillow==11.2.1',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *packages])
import torchvision
print('Existing torchvision:', torchvision.__version__)


## Write the benchmark helper
This cell only writes the helper embedded in this notebook. It does not start GPU inference.


In [ ]:
%%writefile /content/grammarflow_ocr/experiment.py
"""Interactive, bounded Unlimited-OCR benchmark. No public server or tunnels."""
import argparse
import contextlib
import csv
import gc
import io
import json
import math
from pathlib import Path
import re
import subprocess
import sys
import time
import unicodedata

MODEL_ID = 'baidu/Unlimited-OCR-2'
MODEL_REVISION = 'c14071b549f4fcab10909f3c4acc6a2edb366947'
REPO_REVISION = '6c5d1906ba638a7cd87bc00f14452a47fdabf16b'
SAMPLE_TEXT = 'GrammarFlow OCR trial\nInvoice GF-2026-001\nBooks: 12.50\nDelivery: 2.50\nTotal: 15.00\nPlease keep the numbers and punctuation unchanged.'


def normalized(text):
    # Strip only known OCR structural tokens; preserve punctuation and case.
    text = re.sub(r'<\|(?:ref|/ref|det|/det|grounding|eos)\|>|<PAGE>', ' ', text)
    return ' '.join(unicodedata.normalize('NFKC', text).split())


def distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for i, a in enumerate(reference, 1):
        current = [i]
        for j, b in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (a != b)))
        previous = current
    return previous[-1]


def score(reference, hypothesis):
    reference, hypothesis = normalized(reference), normalized(hypothesis)
    if not reference:
        return None
    # Exact edit-distance scoring is intentionally limited to short references.
    # Never truncate the reference and present it as full-document accuracy.
    if len(reference) > 3000 or len(hypothesis) > 6000:
        return {'status': 'not_scored', 'reason': 'Reference/output too long for this small exact scorer.'}
    words = reference.split()
    return {'status': 'scored', 'normalized_CER': distance(reference, hypothesis) / len(reference),
            'normalized_WER': distance(words, hypothesis.split()) / len(words),
            'reference_characters': len(reference), 'normalization': 'NFKC, collapsed whitespace, known structural tokens removed'}


def near_page_cap(token_ids, page_id, limit):
    count = 0
    for token in token_ids:
        count = 0 if token == page_id else count + 1
        if count >= limit - 4:
            return True
    return False


def sample_image(root):
    from PIL import Image, ImageDraw, ImageFont
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    image = Image.new('RGB', (1500, 650), 'white')
    font_path = Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
    font = ImageFont.truetype(str(font_path), 34) if font_path.exists() else ImageFont.load_default(size=34)
    ImageDraw.Draw(image).multiline_text((70, 60), SAMPLE_TEXT, fill='black', font=font, spacing=22)
    target = root / 'synthetic-invoice.png'
    image.save(target)
    return {'id': 'synthetic-invoice', 'path': str(target), 'reference': SAMPLE_TEXT}


def render_pages(path, root, max_pages=3, max_pixels=3000000):
    from PIL import Image, ImageOps
    path, root = Path(path), Path(root)
    root.mkdir(parents=True, exist_ok=True)
    paths, text_layers = [], []
    if path.suffix.lower() == '.pdf':
        import fitz
        with fitz.open(path) as document:
            if document.needs_pass:
                raise ValueError('Password-protected PDFs are not supported in this trial.')
            total_pages = len(document)
            for index in range(min(total_pages, max_pages)):
                page = document[index]
                area = page.rect.width * page.rect.height
                if area <= 0:
                    raise ValueError('Invalid page dimensions.')
                scale = min(2.0, math.sqrt(max_pixels / area))
                target = root / f'page-{index + 1}.png'
                page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False).save(str(target))
                paths.append(str(target))
                text_layers.append(page.get_text())
    else:
        with Image.open(path) as original:
            if original.width * original.height > 40000000:
                raise ValueError('Image exceeds the 40-megapixel trial input limit.')
            image = ImageOps.exif_transpose(original).convert('RGB')
            scale = min(1.0, math.sqrt(max_pixels / (image.width * image.height)))
            if scale < 1:
                image = image.resize((max(1, int(image.width * scale)), max(1, int(image.height * scale))))
            target = root / 'page-1.png'
            image.save(target)
            paths.append(str(target))
            total_pages = 1
    if not paths:
        raise ValueError('No pages available.')
    return {'paths': paths, 'source_pages': total_pages, 'processed_pages': len(paths),
            'pages_omitted': total_pages - len(paths), 'text_layer': '\n'.join(text_layers)}


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)


def worker(config_path):
    import torch
    from transformers import AutoModel, AutoTokenizer
    config = json.loads(Path(config_path).read_text(encoding='utf-8'))
    output = Path(config['output'])
    output.mkdir(parents=True, exist_ok=True)
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('Choose a CUDA GPU with BF16 support, such as an A100. No CPU fallback is run.')
    started = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True, token=False)
    model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True,
                                     use_safetensors=True, torch_dtype=torch.bfloat16,
                                     low_cpu_mem_usage=True, token=False).eval().cuda()
    torch.cuda.synchronize()
    metadata = {'model': MODEL_ID, 'model_revision': MODEL_REVISION, 'repo_revision': REPO_REVISION,
                'gpu': torch.cuda.get_device_name(0), 'gpu_memory_GB': torch.cuda.get_device_properties(0).total_memory / 1e9,
                'torch': torch.__version__, 'cuda': torch.version.cuda,
                'model_load_seconds': round(time.perf_counter() - started, 3),
                'settings': config['limits']}
    write_json(output / 'environment.json', metadata)
    results = []
    for index, item in enumerate(config['documents']):
        target = output / f'document-{index + 1:02d}'
        target.mkdir(exist_ok=True)
        row = {'id': item['id'], 'status': 'failed'}
        try:
            prepared_at = time.perf_counter()
            pages = render_pages(item['path'], target / 'pages', config['limits']['max_pages'])
            (target / 'pdf-text-layer.txt').write_text(pages.pop('text_layer'), encoding='utf-8')
            row.update({k: v for k, v in pages.items() if k != 'paths'})
            row['render_seconds'] = round(time.perf_counter() - prepared_at, 3)
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            inference_at = time.perf_counter()
            with torch.inference_mode(), contextlib.redirect_stdout(io.StringIO()):
                result = model.infer(tokenizer, image_file=pages['paths'], crop_mode=False,
                                     max_new_tokens=config['limits']['tokens_per_page'] * len(pages['paths']),
                                     max_page_tokens=config['limits']['tokens_per_page'], max_crops=4,
                                     eval_mode=True, return_dict=True, save_results=False, temperature=0)
            torch.cuda.synchronize()
            seconds = time.perf_counter() - inference_at
            text = result['text']
            (target / 'ocr-raw.txt').write_text(text, encoding='utf-8')
            row.update(status='completed', inference_seconds=round(seconds, 3),
                       seconds_per_page=round(seconds / len(pages['paths']), 3),
                       output_tokens=len(result['token_ids']), finish_reason=result['finish_reason'],
                       peak_allocated_GB=round(torch.cuda.max_memory_allocated() / 1e9, 3),
                       peak_reserved_GB=round(torch.cuda.max_memory_reserved() / 1e9, 3))
            row['flags'] = []
            if result['finish_reason'] != 'stop':
                row['flags'].append('output_incomplete_or_token_limit')
            if result.get('num_pages_out') != len(pages['paths']):
                row['flags'].append('page_marker_count_mismatch')
            sparse_config = getattr(model.config, 'sparse_attention_config', None) or {}
            page_id = sparse_config.get('page_token_id', 128827) if isinstance(sparse_config, dict) else 128827
            if near_page_cap(result['token_ids'], page_id, config['limits']['tokens_per_page']):
                row['flags'].append('near_per_page_token_cap_review_for_missing_text')
            if row['pages_omitted']:
                row['flags'].append('only_first_pages_processed')
            reference = item.get('reference')
            row['quality'] = score(reference, text) if reference and not row['pages_omitted'] else None
            if reference:
                (target / 'reference.txt').write_text(reference, encoding='utf-8')
            # CER/WER still describes the produced text when incomplete; flags
            # must be considered. It is not an overall layout/semantic score.
        except torch.cuda.OutOfMemoryError:
            row.update(error='GPU out of memory. Reduce pages/resolution and restart the trial.', status='failed')
            row['flags'] = ['out_of_memory']
            results.append(row)
            write_json(output / 'metrics.json', results)
            break
        except Exception as error:
            row['error'] = f'{type(error).__name__}: {str(error)[:300]}'
        results.append(row)
        write_json(output / 'metrics.json', results)
        gc.collect()
        torch.cuda.empty_cache()


def launch(config_path, timeout=600):
    config_path = Path(config_path).resolve()
    config = json.loads(config_path.read_text(encoding='utf-8'))
    if not 1 <= len(config['documents']) <= 3:
        raise ValueError('Trial supports 1–3 documents at a time.')
    if not 1 <= config['limits']['max_pages'] <= 3 or not 128 <= config['limits']['tokens_per_page'] <= 4096:
        raise ValueError('Trial limits out of range.')
    if not 30 <= timeout <= 1200:
        raise ValueError('Timeout must be 30–1200 seconds.')
    output = Path(config['output'])
    output.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    timed_out = False
    with (output / 'worker.log').open('w', encoding='utf-8') as log:
        process = subprocess.Popen([sys.executable, str(Path(__file__).resolve()), '--worker', str(config_path)],
                                   stdout=log, stderr=subprocess.STDOUT)
        try:
            code = process.wait(timeout=timeout)
        except (subprocess.TimeoutExpired, KeyboardInterrupt):
            timed_out = True
            process.kill()
            code = process.wait(timeout=30)
    status = {'exit_code': code, 'timed_out_or_interrupted': timed_out,
              'elapsed_seconds': round(time.perf_counter() - started, 3),
              'note': 'Time limit includes model download/load. Completed document outputs are kept; missing documents were not completed.'}
    write_json(output / 'run-status.json', status)
    return status


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--worker', required=True)
    worker(parser.parse_args().worker)


## Choose a tiny test
Leave `UPLOAD_CUSTOM = False` for the first run. It generates a synthetic invoice with known ground truth.

Later, set it to True and upload up to 3 PDF/PNG/JPEG/WebP files, at most 20 MB each. Optional UTF-8 `.txt` files with matching stems supply human-checked reference text. A PDF reference must correspond exactly to the pages processed; references are not scored when later pages were omitted.

Scanned documents and their OCR outputs stay in this Colab runtime unless you download them. Use non-sensitive examples initially. OCR quality, including Hindi/Hinglish, tables and formulas, is to be measured—not assumed.


In [ ]:
import importlib.util, json, uuid
from google.colab import files

spec = importlib.util.spec_from_file_location('ocr_experiment', ROOT / 'experiment.py')
experiment = importlib.util.module_from_spec(spec)
spec.loader.exec_module(experiment)

UPLOAD_CUSTOM = False
MAX_PAGES_PER_DOCUMENT = 3
TOKENS_PER_PAGE = 2048
MAX_RUN_SECONDS = 600  # Includes model download/load; kills only the benchmark child on timeout.

run_root = ROOT / ('run-' + uuid.uuid4().hex[:10])
run_root.mkdir()
if not UPLOAD_CUSTOM:
    documents = [experiment.sample_image(run_root / 'inputs')]
else:
    uploaded = files.upload()
    supported = {'.pdf', '.png', '.jpg', '.jpeg', '.webp'}
    candidates = [(name, data) for name, data in uploaded.items() if Path(name).suffix.lower() in supported]
    if not 1 <= len(candidates) <= 3:
        raise ValueError('Upload 1–3 supported documents, plus optional matching .txt references.')
    if any(len(data) > 20 * 1024**2 for data in uploaded.values()):
        raise ValueError('Each uploaded file must be at most 20 MB.')
    references = {Path(name).stem: data.decode('utf-8') for name, data in uploaded.items() if Path(name).suffix.lower() == '.txt'}
    documents = []
    for index, (name, data) in enumerate(candidates, 1):
        path = run_root / f'input-{index}{Path(name).suffix.lower()}'
        path.write_bytes(data)  # Never use the uploaded name as a filesystem path.
        documents.append({'id': f'document-{index}', 'path': str(path), 'reference': references.get(Path(name).stem)})
    del uploaded
config = {'output': str(run_root / 'results'), 'documents': documents,
          'limits': {'max_pages': MAX_PAGES_PER_DOCUMENT, 'tokens_per_page': TOKENS_PER_PAGE}}
config_path = run_root / 'config.json'
config_path.write_text(json.dumps(config), encoding='utf-8')
print('Documents:', len(documents), '| Page cap per document:', MAX_PAGES_PER_DOCUMENT)
print('No model inference has started yet. The next cell consumes GPU time.')


## Run the bounded GPU experiment
Weights are loaded once for this run. First load is slower than later inference. The parent process enforces the overall time limit; completed document results survive a timeout. Do not repeatedly retry a failed/OOM run without checking the log and lowering limits.


In [ ]:
status = experiment.launch(config_path, timeout=MAX_RUN_SECONDS)
print(json.dumps(status, indent=2))
results_root = run_root / 'results'
if status['exit_code'] != 0:
    print('The trial did not finish cleanly. Inspect worker.log before spending more compute units.')
    log_path = results_root / 'worker.log'
    if log_path.exists():
        print(log_path.read_text(errors='replace')[-4000:])


## Inspect results and optional compute-unit estimate
CER/WER measure exact text differences against supplied ground truth after documented normalization. They are **not** layout, formula correctness, or overall semantic accuracy scores. Markdown/coordinates can still affect these text metrics. Without a reference, no accuracy percentage is invented. Review the raw output alongside the original page.

`output_incomplete_or_token_limit`, page-marker mismatches, or omitted pages must be considered before judging quality. This small trial does not establish concurrent-user capacity.


In [ ]:
import csv
metrics_path = results_root / 'metrics.json'
rows = json.loads(metrics_path.read_text()) if metrics_path.exists() else []
environment_path = results_root / 'environment.json'
if environment_path.exists():
    print(json.dumps(json.loads(environment_path.read_text()), indent=2))
print(json.dumps(rows, ensure_ascii=False, indent=2))
with (results_root / 'summary.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=['id', 'status', 'processed_pages', 'inference_seconds', 'seconds_per_page', 'peak_allocated_GB', 'finish_reason', 'flags'])
    writer.writeheader()
    for row in rows:
        writer.writerow({key: row.get(key) for key in writer.fieldnames})

CCU_PER_HOUR = None  # Enter the observed consumption rate from your Colab resource panel.
REMAINING_CCU = None  # Enter your actual remaining balance; no account entitlement is assumed.
complete_pages = sum(row.get('processed_pages', 0) for row in rows if row['status'] == 'completed' and not row.get('flags'))
if CCU_PER_HOUR is not None and REMAINING_CCU is not None:
    if CCU_PER_HOUR <= 0 or REMAINING_CCU < 0:
        raise ValueError('Use a positive observed CCU/hour rate and a nonnegative balance.')
    used = CCU_PER_HOUR * status['elapsed_seconds'] / 3600
    estimate = {'approximate_trial_CCU': used,
                'rough_pages_at_remaining_balance': int(complete_pages / used * REMAINING_CCU) if used and complete_pages else None,
                'caution': 'Tiny-sample estimate includes child model load but excludes notebook setup/idle time. It is not a quota or hosting guarantee.'}
    experiment.write_json(results_root / 'compute-estimate.json', estimate)
    print(json.dumps(estimate, indent=2))
else:
    print('No credit-capacity estimate: supply the observed rate and remaining balance first.')


## Download, then disconnect
The archive includes rendered pages, OCR text, references, metrics and the worker log. It can contain your documents: keep it private. Share only non-sensitive metrics when asking for help. Download before disconnecting; runtime files are temporary.


In [ ]:
archive = shutil.make_archive(str(run_root / 'GrammarFlow-OCR-results'), 'zip', results_root)
files.download(archive)


**Finished?** Use **Runtime → Disconnect and delete runtime** now. A loaded or idle GPU can continue consuming your Colab allowance. This notebook does not bypass idle limits or expose the runtime to public users.

Next decision: compare the outputs with GrammarFlow's existing OCR on the same files. Only after quality, page latency and operating cost are measured should we design a production GPU endpoint or paid page credits.
